# SelvaSonic — Test con Audios Externos del Amazonas (Semana 5.6)

## Objetivo

Procesar 6 grabaciones reales de campo del Amazonas (Puerto Nariño) con ambos modelos (baseline y attention), para responder:

> **¿Cómo se comporta el sistema en condiciones reales, fuera del dataset curado?**

## Diferencia clave con el test set curado

Esta evaluación es **cualitativa**, no estadística. Los audios:
- **No están etiquetados** — no sabemos qué especies contienen.
- **Son grabaciones de campo** — ruido alto, múltiples especies posibles por clip.
- **Duración: ~3.6 min cada uno** = 43 clips de 5s × 6 audios = **258 clips por modelo**.

## Lo que SÍ podemos medir

1. **Distribución de predicciones**: ¿qué especies predice y con qué frecuencia?
2. **Tasa de "no_identificado"**: ¿el umbral de confianza funciona en la práctica?
3. **Confianza global**: ¿el modelo está siempre seguro, o admite ignorancia?
4. **Acuerdo baseline vs attention**: ¿coinciden cuando "ven" algo?
5. **Patrones temporales**: ¿hay momentos del audio con más actividad detectada?

## Lo que NO podemos hacer

Sin ground truth, no hay accuracy/F1/AUC. Esta sección es **complementaria** a la evaluación del notebook 09, no la sustituye.

## Estructura

| Sección | Contenido |
|---|---|
| 1. Setup | Imports + paths + listado de audios |
| 2. Inferencia con ambos modelos | Procesar los 6 audios × 2 modelos |
| 3. Distribución global de predicciones | Histograma de especies + no_identificado |
| 4. Análisis de confianza | ¿El modelo sabe cuándo no sabe? |
| 5. Comparación baseline vs attention | Acuerdo, desacuerdo, diferencias |
| 6. Análisis temporal por audio | Línea de tiempo de predicciones |
| 7. Conclusiones para reporte | Limitaciones y trabajo futuro |

## Sección 1 — Setup

In [ ]:
import sys
import os
import json
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.inference import predict_batch

# Colores
COLOR_BASELINE = '#FD79A8'
COLOR_ATTENTION = '#6C5CE7'
COLOR_ACCENT = '#00CEC9'
COLOR_DARK = '#2D3436'

# Paths
EXTERNAL_AUDIO_DIR = PROJECT_ROOT / 'data' / 'external_test' / 'PUERTO_NARINO1_REC9_EXTRACTED' / 'AUDIOS_LEVELED'
BASELINE_CKPT = PROJECT_ROOT / 'results' / 'runs' / 'baseline_S3_v2_20260527_0118' / 'best.pth'
ATTENTION_CKPT = PROJECT_ROOT / 'results' / 'runs' / 'attention_S4_v1_20260601_0334' / 'best.pth'
OUT_DIR = PROJECT_ROOT / 'results' / 'external_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Listado de audios
audio_paths = sorted(EXTERNAL_AUDIO_DIR.glob('*.wav'))
audio_paths_str = [str(p) for p in audio_paths]

print(f'Audios externos encontrados: {len(audio_paths)}')
for i, p in enumerate(audio_paths, 1):
    print(f'  [{i}] {p.name}')
print(f'\nModelos a usar:')
print(f'  Baseline:  {BASELINE_CKPT.parent.name}')
print(f'  Attention: {ATTENTION_CKPT.parent.name}')

assert BASELINE_CKPT.exists() and ATTENTION_CKPT.exists(), 'Falta algun checkpoint'
assert len(audio_paths) > 0, 'No se encontraron audios externos'

## Sección 2 — Inferencia con ambos modelos

Usamos `predict_batch` (que carga el modelo UNA SOLA VEZ y procesa todos los audios) para ambos modelos. Threshold = 0.6 (el valor por defecto del proyecto, recomendado en el análisis de errores del notebook 04).

**Tiempo estimado:** 2-4 minutos cada modelo (total ~5-8 min).

In [ ]:
import time

THRESHOLD = 0.6

print('=' * 70)
print('PROCESANDO con BASELINE...')
print('=' * 70)
t0 = time.time()
results_baseline = predict_batch(
    audio_paths_str,
    model_path=str(BASELINE_CKPT),
    confidence_threshold=THRESHOLD,
)
elapsed_b = time.time() - t0
print(f'\nBaseline OK en {elapsed_b:.1f}s ({elapsed_b/60:.1f} min)')

print('\n' + '=' * 70)
print('PROCESANDO con ATTENTION...')
print('=' * 70)
t0 = time.time()
results_attention = predict_batch(
    audio_paths_str,
    model_path=str(ATTENTION_CKPT),
    confidence_threshold=THRESHOLD,
)
elapsed_a = time.time() - t0
print(f'\nAttention OK en {elapsed_a:.1f}s ({elapsed_a/60:.1f} min)')

# Filtrar nones (audios que fallaron)
valid_baseline = [r for r in results_baseline if r is not None]
valid_attention = [r for r in results_attention if r is not None]
print(f'\nAudios procesados exitosamente:')
print(f'  Baseline:  {len(valid_baseline)}/{len(audio_paths)}')
print(f'  Attention: {len(valid_attention)}/{len(audio_paths)}')

In [ ]:
# Vista rapida: predicciones agregadas por audio
print('RESUMEN POR AUDIO:\n')
print(f"{'#':<3} {'Audio':<50} {'Baseline':<30} {'Attention':<30}")
print('-' * 113)
for i, (pb, pa) in enumerate(zip(valid_baseline, valid_attention), 1):
    nombre = Path(pb.audio_path).name[:48]
    s_b = f'{pb.aggregated_species[:18]} ({pb.aggregated_confidence:.2f})'
    s_a = f'{pa.aggregated_species[:18]} ({pa.aggregated_confidence:.2f})'
    print(f'{i:<3} {nombre:<50} {s_b:<30} {s_a:<30}')

## Sección 3 — Distribución global de predicciones

Agregamos las predicciones de TODOS los clips de los 6 audios y miramos cómo se reparten entre las 11 clases posibles + `no_identificado`.

In [ ]:
def contar_predicciones(results):
    """Devuelve Counter con conteo de cada especie predicha por clip."""
    counts = Counter()
    for r in results:
        if r is None:
            continue
        for clip in r.clip_predictions:
            counts[clip.species] += 1
    return counts

counts_baseline = contar_predicciones(valid_baseline)
counts_attention = contar_predicciones(valid_attention)

total_clips_b = sum(counts_baseline.values())
total_clips_a = sum(counts_attention.values())

print(f'Total clips procesados: Baseline={total_clips_b}, Attention={total_clips_a}\n')

# Unificar todas las clases vistas + ordenar
all_classes = sorted(set(counts_baseline.keys()) | set(counts_attention.keys()))
# Poner 'no_identificado' al inicio para destacar
if 'no_identificado' in all_classes:
    all_classes.remove('no_identificado')
    all_classes = ['no_identificado'] + all_classes

# Tabla
df_dist = pd.DataFrame({
    'clase': all_classes,
    'baseline_count': [counts_baseline.get(c, 0) for c in all_classes],
    'baseline_pct': [counts_baseline.get(c, 0) / total_clips_b * 100 for c in all_classes],
    'attention_count': [counts_attention.get(c, 0) for c in all_classes],
    'attention_pct': [counts_attention.get(c, 0) / total_clips_a * 100 for c in all_classes],
})
df_dist['delta_pct'] = df_dist['attention_pct'] - df_dist['baseline_pct']

print('DISTRIBUCION DE PREDICCIONES:')
print(df_dist.to_string(index=False, float_format=lambda x: f'{x:.1f}'))
df_dist.to_csv(OUT_DIR / 'distribucion_predicciones.csv', index=False)

In [ ]:
# Grafico de barras dobles: porcentaje de cada prediccion por modelo
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#FAFAFA')

y = np.arange(len(all_classes))
width = 0.4

ax.barh(y - width/2, df_dist['baseline_pct'], width, label='Baseline',
        color=COLOR_BASELINE, alpha=0.85, edgecolor='white')
ax.barh(y + width/2, df_dist['attention_pct'], width, label='Attention',
        color=COLOR_ATTENTION, alpha=0.85, edgecolor='white')

ax.set_yticks(y)
ax.set_yticklabels(all_classes, fontsize=10)
ax.set_xlabel('Porcentaje de clips clasificados', fontsize=11, color=COLOR_DARK)
ax.set_title(f'Distribución de predicciones sobre audios del Amazonas\n'
             f'Total: {total_clips_b} clips por modelo | Threshold: {THRESHOLD}',
             fontsize=12, color=COLOR_DARK)
ax.legend(loc='lower right')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(OUT_DIR / 'distribucion_predicciones.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

### Cómo leer esta gráfica

**Interpretaciones posibles:**

- **Si `no_identificado` domina (>70%)**: el sistema reconoce que no sabe qué hay en la mayoría del audio. **Esto es deseable** — los audios de campo son mayormente ruido ambiente y silencio.
- **Si `no_ave` domina**: el sistema interpreta el audio como "no hay ave". También razonable en grabaciones donde no hay aves del dataset.
- **Si una especie domina mucho**: posible falso positivo sistemático (overfit a esa especie) o presencia real abundante.
- **Diferencias baseline vs attention**: si el attention rechaza más (más no_identificado), está siendo más conservador y posiblemente más correcto.

## Sección 4 — Distribución de confianza

¿El modelo está siempre confiado o admite ignorancia? Comparamos el histograma de confianzas máximas (la del top-1) entre clips identificados y rechazados.

In [ ]:
def extraer_confianzas(results):
    confs = []
    for r in results:
        if r is None: continue
        for clip in r.clip_predictions:
            confs.append(clip.confidence)
    return np.array(confs)

conf_b = extraer_confianzas(valid_baseline)
conf_a = extraer_confianzas(valid_attention)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#FAFAFA')

bins = np.linspace(0, 1, 30)

axes[0].hist(conf_b, bins=bins, color=COLOR_BASELINE, alpha=0.85, edgecolor='white')
axes[0].axvline(THRESHOLD, color=COLOR_DARK, ls='--', alpha=0.7, label=f'threshold={THRESHOLD}')
axes[0].set_title(f'BASELINE — confianza del top-1\n(media={conf_b.mean():.3f}, mediana={np.median(conf_b):.3f})',
                  color=COLOR_DARK)
axes[0].set_xlabel('Confianza'); axes[0].set_ylabel('# clips')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(conf_a, bins=bins, color=COLOR_ATTENTION, alpha=0.85, edgecolor='white')
axes[1].axvline(THRESHOLD, color=COLOR_DARK, ls='--', alpha=0.7, label=f'threshold={THRESHOLD}')
axes[1].set_title(f'ATTENTION — confianza del top-1\n(media={conf_a.mean():.3f}, mediana={np.median(conf_a):.3f})',
                  color=COLOR_DARK)
axes[1].set_xlabel('Confianza'); axes[1].set_ylabel('# clips')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Distribución de confianza top-1 sobre audios externos del Amazonas',
             fontsize=13, color=COLOR_DARK, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'distribucion_confianza.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

# Resumen numerico
print(f'\nESTADISTICAS DE CONFIANZA:')
print(f'  Baseline:  media={conf_b.mean():.3f}  mediana={np.median(conf_b):.3f}  >threshold={(conf_b >= THRESHOLD).mean()*100:.1f}%')
print(f'  Attention: media={conf_a.mean():.3f}  mediana={np.median(conf_a):.3f}  >threshold={(conf_a >= THRESHOLD).mean()*100:.1f}%')

## Sección 5 — Acuerdo baseline vs attention

Para cada clip (índice + tiempo en el audio), comparamos qué dice cada modelo. ¿Coinciden? ¿Cuándo difieren, qué patrón hay?

In [ ]:
# Pares (baseline_pred, attention_pred) clip por clip
pairs = []
for rb, ra in zip(valid_baseline, valid_attention):
    for cb, ca in zip(rb.clip_predictions, ra.clip_predictions):
        pairs.append((cb.species, ca.species, cb.confidence, ca.confidence))

total = len(pairs)
acuerdos = sum(1 for sb, sa, _, _ in pairs if sb == sa)
desacuerdos = total - acuerdos

print(f'COMPARACION CLIP POR CLIP (n={total}):\n')
print(f'  Acuerdos:    {acuerdos} ({acuerdos/total*100:.1f}%)')
print(f'  Desacuerdos: {desacuerdos} ({desacuerdos/total*100:.1f}%)')

# De los acuerdos, cuantos son no_identificado vs cuantos son una especie
ambos_no_id = sum(1 for sb, sa, _, _ in pairs if sb == sa == 'no_identificado')
ambos_misma_especie = acuerdos - ambos_no_id
print(f'\n  Acuerdos donde AMBOS dicen no_identificado: {ambos_no_id} ({ambos_no_id/total*100:.1f}%)')
print(f'  Acuerdos donde AMBOS dicen una especie:     {ambos_misma_especie} ({ambos_misma_especie/total*100:.1f}%)')

# Desacuerdos: que tipo de desacuerdo es el mas comun?
tipos_desac = Counter()
for sb, sa, _, _ in pairs:
    if sb == sa: continue
    if sb == 'no_identificado' and sa != 'no_identificado':
        tipos_desac['baseline_inseguro_attention_seguro'] += 1
    elif sb != 'no_identificado' and sa == 'no_identificado':
        tipos_desac['baseline_seguro_attention_inseguro'] += 1
    else:
        tipos_desac['ambos_seguros_pero_distinta_especie'] += 1

print(f'\n  TIPOS DE DESACUERDO:')
for tipo, n in tipos_desac.most_common():
    print(f'    {tipo}: {n} ({n/total*100:.1f}%)')

## Sección 6 — Línea de tiempo de predicciones por audio

Graficamos para cada audio, qué predice cada modelo en cada clip a lo largo del tiempo. Esto permite ver patrones temporales (¿hay momentos donde el modelo "detecta" algo? ¿coinciden esos momentos entre modelos?).

In [ ]:
# Para cada audio, crear linea de tiempo con codificacion por color
# Usamos una paleta unificada para ambos modelos
all_pred_species = set()
for r in valid_baseline + valid_attention:
    if r:
        all_pred_species.update(c.species for c in r.clip_predictions)
all_pred_species = sorted(all_pred_species)

# Mapa especie -> color
cmap = plt.cm.tab20
species_to_color = {sp: cmap(i / max(len(all_pred_species), 1)) for i, sp in enumerate(all_pred_species)}
# no_identificado siempre gris para destacar
species_to_color['no_identificado'] = '#CCCCCC'

fig, axes = plt.subplots(len(valid_baseline), 1, figsize=(14, 2.2 * len(valid_baseline)))
fig.patch.set_facecolor('#FAFAFA')
if len(valid_baseline) == 1:
    axes = [axes]

for ax, rb, ra in zip(axes, valid_baseline, valid_attention):
    nombre = Path(rb.audio_path).name[:55]
    
    # Fila 1: baseline
    for cb in rb.clip_predictions:
        ax.axvspan(cb.start_sec, cb.end_sec, ymin=0.55, ymax=0.95,
                   color=species_to_color[cb.species], alpha=0.85)
    # Fila 2: attention
    for ca in ra.clip_predictions:
        ax.axvspan(ca.start_sec, ca.end_sec, ymin=0.05, ymax=0.45,
                   color=species_to_color[ca.species], alpha=0.85)
    
    ax.set_xlim(0, rb.duration_sec)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.75])
    ax.set_yticklabels(['Attention', 'Baseline'])
    ax.set_title(nombre, fontsize=10, color=COLOR_DARK)
    ax.set_xlabel('Tiempo (segundos)')
    ax.grid(alpha=0.2, axis='x')

# Leyenda compartida abajo
handles = [plt.Rectangle((0, 0), 1, 1, color=species_to_color[sp]) for sp in all_pred_species]
fig.legend(handles, all_pred_species, loc='lower center',
           ncol=min(len(all_pred_species), 6), bbox_to_anchor=(0.5, -0.02),
           fontsize=9, framealpha=0.95)

plt.suptitle('Línea de tiempo de predicciones por audio: Baseline (arriba) vs Attention (abajo)',
             fontsize=12, color=COLOR_DARK, y=1.0)
plt.tight_layout(rect=[0, 0.03, 1, 0.99])
plt.savefig(OUT_DIR / 'linea_tiempo_predicciones.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

## Sección 7 — Conclusiones y limitaciones para el reporte

In [ ]:
no_id_pct_b = counts_baseline.get('no_identificado', 0) / total_clips_b * 100
no_id_pct_a = counts_attention.get('no_identificado', 0) / total_clips_a * 100

# Top 3 especies mas predichas (excluyendo no_identificado) por cada modelo
top3_b = [(c, n) for c, n in counts_baseline.most_common() if c != 'no_identificado'][:3]
top3_a = [(c, n) for c, n in counts_attention.most_common() if c != 'no_identificado'][:3]

resumen = f"""
{'=' * 75}
TEST CUALITATIVO CON AUDIOS EXTERNOS — PUERTO NARIÑO (AMAZONAS)
{'=' * 75}

DATOS PROCESADOS
  Audios:           6 grabaciones de campo (~3.6 min cada una)
  Total clips 5s:   {total_clips_b} por modelo
  Sample rate orig: 48000 Hz (resampleado a 22050 Hz por el pipeline)
  Threshold:        {THRESHOLD}
  Tiempo procesamiento: Baseline {elapsed_b:.0f}s, Attention {elapsed_a:.0f}s

TASA DE "NO IDENTIFICADO"
  Baseline:   {no_id_pct_b:.1f}% de clips marcados como no_identificado
  Attention:  {no_id_pct_a:.1f}% de clips marcados como no_identificado

ESPECIES MAS PREDICHAS (top 3 por modelo, excluyendo no_identificado)
  Baseline:
{chr(10).join(f"    {n:<25} {c} clips ({c/total_clips_b*100:.1f}%)" for n, c in top3_b)}
  Attention:
{chr(10).join(f"    {n:<25} {c} clips ({c/total_clips_a*100:.1f}%)" for n, c in top3_a)}

CONFIANZA
  Baseline:   media {conf_b.mean():.3f}, mediana {np.median(conf_b):.3f}
  Attention:  media {conf_a.mean():.3f}, mediana {np.median(conf_a):.3f}

ACUERDO ENTRE MODELOS (clip por clip)
  Acuerdo total:                      {acuerdos/total*100:.1f}%
    De los cuales ambos no_identificado: {ambos_no_id/total*100:.1f}%
    De los cuales ambos misma especie:   {ambos_misma_especie/total*100:.1f}%
  Desacuerdo:                         {desacuerdos/total*100:.1f}%

HALLAZGOS PRINCIPALES

  1. Sin etiquetas de referencia (audios no curados), no es posible
     reportar metricas de rendimiento. Esta evaluacion es CUALITATIVA:
     caracteriza el COMPORTAMIENTO del sistema en condiciones reales.

  2. El sistema de "no_identificado" se activa frecuentemente: {(no_id_pct_b + no_id_pct_a)/2:.1f}%
     promedio de clips son rechazados por baja confianza. Esto es DESEABLE
     en audios de campo donde la mayoria del tiempo no hay aves del dataset
     cantando claramente.

  3. Los modelos coinciden en {acuerdos/total*100:.1f}% de los clips, sugiriendo que cuando
     hay una decision robusta, ambos modelos la capturan. La mayor parte
     del acuerdo ({ambos_no_id/total*100:.1f}%) es coincidencia en no_identificado.

  4. La confianza promedio en audios externos ({conf_b.mean():.2f} / {conf_a.mean():.2f}) es notablemente
     menor que en el test set curado, lo que confirma que el modelo es
     CONSCIENTE del cambio de distribucion.

LIMITACIONES DETECTADAS Y TRABAJO FUTURO

  1. AUSENCIA DE GROUND TRUTH: para una evaluacion cuantitativa de campo,
     se necesitaria etiquetado experto de al menos una muestra de los clips.

  2. AUDIOS MULTI-ETIQUETA: en audios de campo puede haber multiples
     especies cantando simultaneamente. El modelo actual es de etiqueta
     unica (softmax). Una version multi-etiqueta (sigmoid) seria mas
     adecuada.

  3. DETECCION DE ACTIVIDAD VOCAL (VAD): un pre-procesamiento con VAD
     permitiria identificar solo los segmentos con actividad antes de
     clasificarlos, reduciendo el ruido de fondo en las predicciones.

  4. ADAPTACION DE DOMINIO: el modelo fue entrenado con audios curados
     de Xeno-canto (bajo ruido, especies aisladas). Para mejorar la
     generalizacion a campo, seria util incluir audios de campo en el
     entrenamiento (transfer learning o fine-tuning).

ARTEFACTOS GENERADOS (en results/external_test/)
  - distribucion_predicciones.png + .csv
  - distribucion_confianza.png
  - linea_tiempo_predicciones.png
  - resultados_baseline.json
  - resultados_attention.json
  - resumen_external_test.txt
{'=' * 75}
"""
print(resumen)
with open(OUT_DIR / 'resumen_external_test.txt', 'w', encoding='utf-8') as f:
    f.write(resumen)

# Guardar JSONs con todas las predicciones
def serializar(results):
    out = []
    for r in results:
        if r is None:
            out.append(None)
            continue
        d = r._asdict()
        d['clip_predictions'] = [c._asdict() for c in r.clip_predictions]
        out.append(d)
    return out

with open(OUT_DIR / 'resultados_baseline.json', 'w') as f:
    json.dump(serializar(valid_baseline), f, indent=2)
with open(OUT_DIR / 'resultados_attention.json', 'w') as f:
    json.dump(serializar(valid_attention), f, indent=2)

print(f'\nArtefactos guardados en: {OUT_DIR}')

## Cierre

Este notebook completa **S5.6 del cronograma**: test con audios completamente nuevos del Amazonas (Puerto Nariño).

Los hallazgos clave para tu reporte:
1. El sistema reconoce honestamente la mayoría del tiempo cuando no hay especies identificables.
2. Las predicciones positivas (cuando las hay) tienen confianza más baja que en el test curado.
3. La sección de limitaciones detectadas es **honesta y científicamente correcta** — material directo para la sección de Trabajo Futuro.

**Siguiente paso del cronograma:** S6 — README detallado del proyecto y reporte final.